# Spike — MERGE INTO isolado (erp_lotes_producao)

Testa a transformação Bronze -> Silver: limpeza via UDF (reaproveitando limpeza_utils.py testado isoladamente) e gravação idempotente via MERGE INTO por chave natural (lote_id).

Referências: ADR-002 (streaming só Landing→Bronze; batch+MERGE daqui em diante).

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DateType, FloatType, StringType, IntegerType
from src.transformacao.limpeza_utils import parse_data_suja, parse_numero_sujo, limpar_texto

udf_data = udf(parse_data_suja, DateType())
udf_numero = udf(parse_numero_sujo, FloatType())
udf_texto = udf(limpar_texto, StringType())

df_bronze = spark.table("poc_pulse_observability.bronze.erp_lotes_producao")

df_silver = (
    df_bronze
    .withColumn("centro_producao_id", udf_texto("centro_producao_id"))
    .withColumn("data_fabricacao", udf_data("data_fabricacao"))
    .withColumn("data_validade", udf_data("data_validade"))
    .withColumn("quantidade_produzida", udf_numero("quantidade_produzida").cast(IntegerType()))
    .withColumn("status_qc", udf_texto("status_qc"))
    .withColumn("data_liberacao", udf_data("data_liberacao"))
    .withColumnRenamed("data", "data_particao_ingestao")
)

df_silver.printSchema()
df_silver.show(5, truncate=False)

In [0]:
df_silver.filter(df_silver.status_qc == "reprovado").select("lote_id", "status_qc", "data_liberacao").show(5)

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.silver.erp_lotes_producao")

print("Total gravado:", spark.table("poc_pulse_observability.silver.erp_lotes_producao").count())

In [0]:
from delta.tables import DeltaTable

tabela_silver = DeltaTable.forName(spark, "poc_pulse_observability.silver.erp_lotes_producao")

(
    tabela_silver.alias("silver")
    .merge(df_silver.alias("bronze_limpo"), "silver.lote_id = bronze_limpo.lote_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Total após MERGE:", spark.table("poc_pulse_observability.silver.erp_lotes_producao").count())

In [0]:
from src.transformacao.configuracao_tabelas import CONFIGURACAO_TABELAS
from src.transformacao.transformar_bronze_para_silver import transformar_bronze_para_silver

resultado = transformar_bronze_para_silver(
    spark=spark,
    catalog="poc_pulse_observability",
    tabela="erp_lotes_producao",
    config=CONFIGURACAO_TABELAS["erp_lotes_producao"],
)
print(resultado)

In [0]:
resultado_crm = transformar_bronze_para_silver(
    spark=spark,
    catalog="poc_pulse_observability",
    tabela="crm_pedidos",
    config=CONFIGURACAO_TABELAS["crm_pedidos"],
)
print(resultado_crm)

In [0]:
resultado_crm = transformar_bronze_para_silver(
    spark=spark,
    catalog="poc_pulse_observability",
    tabela="crm_pedidos",
    config=CONFIGURACAO_TABELAS["crm_pedidos"],
)
print(resultado_crm)